In [1]:
"""
The purpose of this Juyter notebook is to format the training and
validation data such that it can be directly passed to the training
script.
"""

'\nThe purpose of this Juyter notebook is to format the training and\nvalidation data such that it can be directly passed to the training\nscript.\n'

In [5]:
import os
import ast

import numpy as np
import pandas as pd

# Create Subdirectories

In [6]:
# Create a subdirectory for the P:U ratio of 1:10
pu_1_10_formatted_data_dir = "formatted_data_PU_ratio_1_10"

if not os.path.exists(pu_1_10_formatted_data_dir):
    os.makedirs(pu_1_10_formatted_data_dir)

In [7]:
# Create a subdirectory for the P:U ratio of 1:5
pu_1_5_formatted_data_dir = "formatted_data_PU_ratio_1_5"

if not os.path.exists(pu_1_5_formatted_data_dir):
    os.makedirs(pu_1_5_formatted_data_dir)

# Data Formatting for P:U Ratio of 1:10

## Loading Data

In [13]:
# Load the training set
training_set_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_10/PU_training_set.tsv"
)

training_set_df = pd.read_csv(
    training_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the validation set
val_set_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_10/PU_validation_set.tsv"
)

val_set_df = pd.read_csv(
    val_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the PPI probability data
ppi_preds_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Inference_on_Dharmacon_pooled_G1_G2_screening_plates_"
    "subset/all_ppi_predictions.tsv"
)

ppi_preds_df = pd.read_csv(
    ppi_preds_path,
    sep="\t"
)

## Formatting the Data

In [21]:
# For the sake of convenience, define a function performing data
# formatting
def format_data(data_set_df, ppi_df, ppi_feature="interaction_probability"):
    """
    Format training and validation data such that it can be fed into the
    PU learning architecture.

    Parameters
    ----------
    data_set_df: Pandas DataFrame
        ...
    ppi_df: Pandas DataFrame
        ...
    ppi_feature: str, default="interaction_probability"
        A string indicating whether to use probabilities or logits for
        the `ppi_vec` feature. Valid options are
        "interaction_probability" and "logits".
    """
    # Create a dictionary mapping UniProt accessions to their
    # corresponding PPI probabiity vector of length 440 (or PPI logit
    # vector)
    prot_to_ppi_prob_vector_dict = {}

    # To this end, the `.groupby()` method is employed
    for protein, group in ppi_df.groupby(by="protein_1"):
        ppi_prob_vector = group[ppi_feature].to_numpy()
        assert ppi_prob_vector.shape[0] == 440, (
            f"For protein {protein}, some PPI probabilities are missing!"
        )
        prot_to_ppi_prob_vector_dict[protein] = ppi_prob_vector

    # The probabilities must first be aggregated as some genes encode
    # multiple proteins
    # Create a dictionary mapping gene names to the proteins they encode
    # Conveniently enough, multiple occurrences of individual genes have
    # already been collaped and the proteins encoded by the individual
    # genes are stored as lists in the `protein_ids` column
    gene_to_prot_dict = (
        data_set_df.set_index("gene")["protein_ids"].to_dict()
    )

    # For each gene, aggregate the corresponding PPI probability vectors
    agg_ppi_prob_vector_dict = {}

    for gene, prot_list in gene_to_prot_dict.items():
        ppi_prob_vector_list = [
            prot_to_ppi_prob_vector_dict[protein]
            for protein in prot_list
        ]
        agg_ppi_prob_vector_dict[gene] = np.max(
            ppi_prob_vector_list, axis=0
        ).tolist()
    
    formatted_df = data_set_df.copy()
    
    # Add a column to the `data_set_df` DataFrame storing the aggregated
    # PPI probability vectors
    # The aggregated vectors have to be arranged in the correct order
    formatted_df["ppi_vec"] = data_set_df["gene"].apply(
        lambda gene: agg_ppi_prob_vector_dict[gene]
    )

    # Retain and reorder columns of interest
    formatted_df = formatted_df[[
        "gene", "phenotype_vec", "ppi_vec", "label"
    ]]

    return formatted_df

In [22]:
# Format the training and validation set using probabilities as PPI
# feature and save them to disk
formatted_training_df = format_data(
    training_set_df, ppi_preds_df
)

formatted_training_df.to_csv(
    os.path.join(
        pu_1_10_formatted_data_dir,
        "training_set_formatted_with_probs_pu_ratio_1_10.tsv"
    ),
    sep="\t",
    index=False
)

formatted_val_df = format_data(
    val_set_df, ppi_preds_df
)

formatted_val_df.to_csv(
    os.path.join(
        pu_1_10_formatted_data_dir,
        "val_set_formatted_with_probs_pu_ratio_1_10.tsv"
    ),
    sep="\t",
    index=False
)

In [23]:
# Format the training and validation set using logits as PPI feature and
# save them to disk
formatted_training_df = format_data(
    training_set_df, ppi_preds_df, ppi_feature="logits"
)

formatted_training_df.to_csv(
    os.path.join(
        pu_1_10_formatted_data_dir,
        "training_set_formatted_with_logits_pu_ratio_1_10.tsv"
    ),
    sep="\t",
    index=False
)

formatted_val_df = format_data(
    val_set_df, ppi_preds_df, ppi_feature="logits"
)

formatted_val_df.to_csv(
    os.path.join(
        pu_1_10_formatted_data_dir,
        "val_set_formatted_with_logits_pu_ratio_1_10.tsv"
    ),
    sep="\t",
    index=False
)

# Data Formatting for P:U Ratio of 1:5

## Loading Data

In [26]:
# Load the training set
training_set_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_5/PU_training_set.tsv"
)

training_set_df = pd.read_csv(
    training_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the validation set
val_set_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_5/PU_validation_set.tsv"
)

val_set_df = pd.read_csv(
    val_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

# Load the PPI probability data
ppi_preds_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Inference_on_Dharmacon_pooled_G1_G2_screening_plates_"
    "subset/all_ppi_predictions.tsv"
)

ppi_preds_df = pd.read_csv(
    ppi_preds_path,
    sep="\t"
)

## Formatting the Data

In [27]:
# Format the training and validation set using probabilities as PPI
# feature and save them to disk
formatted_training_df = format_data(
    training_set_df, ppi_preds_df
)

formatted_training_df.to_csv(
    os.path.join(
        pu_1_5_formatted_data_dir,
        "training_set_formatted_with_probs_pu_ratio_1_5.tsv"
    ),
    sep="\t",
    index=False
)

formatted_val_df = format_data(
    val_set_df, ppi_preds_df
)

formatted_val_df.to_csv(
    os.path.join(
        pu_1_5_formatted_data_dir,
        "val_set_formatted_with_probs_pu_ratio_1_5.tsv"
    ),
    sep="\t",
    index=False
)

In [28]:
# Format the training and validation set using logits as PPI feature and
# save them to disk
formatted_training_df = format_data(
    training_set_df, ppi_preds_df, ppi_feature="logits"
)

formatted_training_df.to_csv(
    os.path.join(
        pu_1_5_formatted_data_dir,
        "training_set_formatted_with_logits_pu_ratio_1_5.tsv"
    ),
    sep="\t",
    index=False
)

formatted_val_df = format_data(
    val_set_df, ppi_preds_df, ppi_feature="logits"
)

formatted_val_df.to_csv(
    os.path.join(
        pu_1_5_formatted_data_dir,
        "val_set_formatted_with_logits_pu_ratio_1_5.tsv"
    ),
    sep="\t",
    index=False
)

# Shuffling Labels

In [30]:
# The labels of the training data set are shuffled in order to
# investigate whether the tranining data set contains learnable signal
# at all
# This is done for the P:U ratio of 1:5
formatted_training_df_pu_ratio_1_5 = pd.read_csv(
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/data_formatting/formatted_"
    "data_PU_ratio_1_5/training_set_formatted_with_logits_pu_ratio_1_"
    "5.tsv",
    sep="\t"
)

formatted_training_df_pu_ratio_1_5["label"] = (
    formatted_training_df_pu_ratio_1_5["label"]
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

In [32]:
formatted_training_df_pu_ratio_1_5.to_csv(
    os.path.join(
        pu_1_5_formatted_data_dir,
        "training_set_formatted_with_logits_pu_ratio_1_5_labels_shuffled.tsv.tsv"
    ),
    sep="\t",
    index=False
)